In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Import and read two point csv files for passing, rushing, receiving, and defensive stats
two_point_pass = pd.read_csv('twoPtPass.csv')
two_point_rush = pd.read_csv('twoPtRush.csv')
two_point_rec = pd.read_csv('twoPtRec.csv')
two_point_def = pd.read_csv('twoPtDef.csv')

In [18]:
two_point_pass.head()
two_point_rush.head()
two_point_rec.head()
two_point_def.head()

,twoPtDefId,playId,teamId,playerId,twoPtDefPosition,twoPtDefResult,twoPtDefNull
0,11430001,11017079,1400,20140031,CB,failed,0
1,11430002,11018356,1400,20110460,CB,def attempt,0
2,11430003,11029760,325,20140079,DB,def attempt,1
3,11430004,11035262,3300,20150031,LB,def conversion,0
4,11430005,11035985,3300,20130015,DB,failed,0


In [19]:
#Check if missing data or null values are present in dataset(One in two_point_rec)
two_point_pass.isnull().sum()
two_point_rush.isnull().sum()
two_point_def.isnull().sum()
two_point_rec.isnull().sum()

twoPtRecId             0
playId                 0
teamId                 0
playerId               0
twoPointRecPosition    0
twoPtRecResult         0
twoPtRecComp           1
twoPtRecNull           0
dtype: int64

In [20]:
#Drop null row in dataset
two_point_rec = two_point_rec.dropna(subset=['twoPtRecComp'])

In [21]:
#Clean Data by dropping the null columns, playId, teamId, and platerId from the dataframes
two_point_def = two_point_def.drop(columns=['twoPtDefNull','teamId','playerId','twoPtDefId'])
two_point_rush = two_point_rush.drop(columns=['twoPtRushNull','teamId','playerId','twoPtRushId'])
two_point_pass = two_point_pass.drop(columns=['twoPtPassNull','teamId','playerId','twoPtPassId'])
two_point_rec = two_point_rec.drop(columns=['twoPtRecNull','teamId','playerId','twoPtRecId'])


In [22]:
#Combine two_point_pass and two_point_rec using playId
two_point_pass_rec = two_point_pass.merge(
    two_point_rec,
    on='playId',
    how='left',
    suffixes=('_pass', '_rec')
)
two_point_pass_rec

#Clean Data
two_point_pass_rec = two_point_pass_rec.drop(columns=['twoPtRecResult','twoPtRecComp'])
two_point_pass_rec


,playId,twoPtPassPosition,twoPtPassConv,twoPtPassSack,twoPtPassComp,twoPointRecPosition
0,115,QB,failed,0,0,RB
1,2564,QB,good,0,1,WR
2,3999,QB,good,0,1,WR
3,4990,QB,failed,0,0,WR
4,7442,QB,good,0,1,TE
...,...,...,...,...,...,...
1212,15079041,QB,good,0,1,WR
1213,15079221,QB,good,0,1,WR
1214,15082774,QB,failed,0,0,WR
1215,15083215,QB,good,0,1,WR


In [23]:
# See rows where receiver info is missing
missing_receivers = two_point_pass_rec[
    two_point_pass_rec['twoPointRecPosition'].isnull()
]

#Analyze the Information
missing_receivers
missing_receivers[['twoPtPassConv', 'twoPtPassSack', 'twoPtPassComp']].value_counts()

twoPtPassConv  twoPtPassSack  twoPtPassComp
failed         1              0                33
               0              0                29
                              1                 3
Name: count, dtype: int64

In [24]:
#New Columnn with target status
two_point_pass_rec['target_status'] = np.select(
    [
        two_point_pass_rec['twoPtPassSack'] == 1,
        two_point_pass_rec['twoPointRecPosition'].isnull()
    ],
    [
        'Sacked',
        'No target recorded'
    ],
    default='Target recorded'
)

#Check Dataset
two_point_pass_rec

,playId,twoPtPassPosition,twoPtPassConv,twoPtPassSack,twoPtPassComp,twoPointRecPosition,target_status
0,115,QB,failed,0,0,RB,Target recorded
1,2564,QB,good,0,1,WR,Target recorded
2,3999,QB,good,0,1,WR,Target recorded
3,4990,QB,failed,0,0,WR,Target recorded
4,7442,QB,good,0,1,TE,Target recorded
...,...,...,...,...,...,...,...
1212,15079041,QB,good,0,1,WR,Target recorded
1213,15079221,QB,good,0,1,WR,Target recorded
1214,15082774,QB,failed,0,0,WR,Target recorded
1215,15083215,QB,good,0,1,WR,Target recorded


In [25]:
#Delete Unnecessary Columns
two_point_pass_rec = two_point_pass_rec.drop(columns=['twoPtPassSack','twoPtPassComp'])
two_point_pass_rec

,playId,twoPtPassPosition,twoPtPassConv,twoPointRecPosition,target_status
0,115,QB,failed,RB,Target recorded
1,2564,QB,good,WR,Target recorded
2,3999,QB,good,WR,Target recorded
3,4990,QB,failed,WR,Target recorded
4,7442,QB,good,TE,Target recorded
...,...,...,...,...,...
1212,15079041,QB,good,WR,Target recorded
1213,15079221,QB,good,WR,Target recorded
1214,15082774,QB,failed,WR,Target recorded
1215,15083215,QB,good,WR,Target recorded


In [26]:
two_point_rush

,playId,twoPtRushPosition,twoPtRushResult
0,1262,QB,good
1,6529,QB,failed
2,9417,RB,failed
3,15440,P,failed
4,22565,QB,good
...,...,...,...
430,15073889,RB,good
431,15074481,RB,good
432,15075946,QB,failed
433,15078896,QB,good


In [27]:
pip install psycopg2-binary sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [31]:
from sqlalchemy import create_engine

# Step 1: Connect to PostgreSQL
# Replace placeholders with your actual details
username = "postgres"      # default user
password = "07032006Ez$/" # the password you set during installation
host = "localhost"         # if running locally
port = "5432"              # default PostgreSQL port
database = "NFLTWOPOINTPASSREC"    # the database you created in pgAdmin

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

# Step 2: Load DataFrame into PostgreSQL
table_name = "twopointpassrec"   # choose any table name
two_point_pass_rec.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

Data successfully loaded into table 'twopointpassrec' in database 'NFLTWOPOINTPASSREC'.


In [32]:
# Step 1: Connect to PostgreSQL
# Replace placeholders with your actual details
username = "postgres"      # default user
password = "07032006Ez$/" # the password you set during installation
host = "localhost"         # if running locally
port = "5432"              # default PostgreSQL port
database = "NFLTWOPOINTPASSREC"    # the database you created in pgAdmin

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

# Step 2: Load DataFrame into PostgreSQL
table_name = "twopointrush"   # choose any table name
two_point_rush.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

Data successfully loaded into table 'twopointrush' in database 'NFLTWOPOINTPASSREC'.
